# Joined matches and members

This notebook checks assumptions that depend on both training files: whether IDs line up, whether there is any reusable history, and how the observed clan points relate to the current member snapshot.


In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 120)

PROJECT_ROOT = Path("../..").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

TRAIN_DATA_PATH = PROJECT_ROOT / "data" / "train"

members = pd.read_csv(TRAIN_DATA_PATH / "member_stats_training.csv")
matches = pd.read_csv(TRAIN_DATA_PATH / "clan_matches_training.csv")


print("members:", members.shape)
print("matches:", matches.shape)

display(members.sample(5))
display(matches.sample(5))

members: (291456, 15)
matches: (24288, 5)


,clan_id,user_id,days_active_last_28_days,days_active_last_7_days,days_since_last_active,registration_country,registration_platform_specific,cohort_day,training_count_last_28_days,is_payer_lifetime,dynamic_payment_segment,clan_multiplier,avg_stars_top_11_players,avg_stars_top_3_players,avg_training_bonus
247296,clan_47281,user_247297,28,7,0,Thailand,Android Phone,410,106,False,0) NonPayer,3,6.585770,7.311111,13.0
218072,clan_41380,user_218073,25,7,0,Morocco,Android Phone,418,30,False,0) NonPayer,3,6.026945,6.821333,12.0
174247,clan_1536,user_174248,26,7,0,Argentina,Flash FB Canvas,4673,265,False,0) NonPayer,2,6.173212,6.947511,11.0
103763,clan_22993,user_103764,28,7,0,Sweden,iOS Phone,63,49,False,0) NonPayer,3,5.497176,5.924445,7.5
61,clan_62,user_62,28,7,0,Australia,iOS Tablet,526,97,True,4) Whale,3,8.904364,10.160267,17.0


,clan_1_id,clan_2_id,clan_1_points,clan_2_points,clan_winner
21279,clan_10586,clan_8233,69,30,1
22581,clan_34581,clan_24732,75,29,1
1160,clan_19761,clan_46768,18,72,2
36,clan_18435,clan_36103,0,102,2
22212,clan_12923,clan_28870,74,21,1


## Minimal cleanup

Only the fields used in this notebook are cleaned here. The negative-star anomaly was already inspected in the member EDA; 


In [14]:
members = members.copy()
matches = matches.copy()

numeric_member_cols = [
    "days_active_last_28_days",
    "days_active_last_7_days",
    "days_since_last_active",
    "cohort_day",
    "training_count_last_28_days",
    "clan_multiplier",
    "avg_stars_top_11_players",
    "avg_stars_top_3_players",
    "avg_training_bonus",
]

for col in numeric_member_cols:
    members[col] = pd.to_numeric(members[col], errors="coerce")

members["avg_stars_top_11_players"] = members["avg_stars_top_11_players"].clip(lower=0)

members["is_payer_lifetime_num"] = (
    members["is_payer_lifetime"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map({"true": 1, "false": 0})
)

matches["point_diff"] = matches["clan_1_points"] - matches["clan_2_points"]
matches["abs_point_diff"] = matches["point_diff"].abs()

print("cleanup complete")

cleanup complete


## Cross-file coverage

This repeats the most important join check from the member notebook in one place: every clan appearing in a match should have member rows, and the expected roster size is six.


In [15]:
match_clans = pd.Index(
    pd.concat([matches["clan_1_id"], matches["clan_2_id"]]).unique(),
    name="clan_id",
)
member_clans = pd.Index(members["clan_id"].unique(), name="clan_id")

missing_in_members = match_clans.difference(member_clans)
extra_in_members = member_clans.difference(match_clans)

coverage = pd.DataFrame(
    {
        "metric": [
            "unique clans in matches",
            "unique clans in members",
            "match clans missing in members",
            "member clans not used in matches",
        ],
        "count": [
            len(match_clans),
            len(member_clans),
            len(missing_in_members),
            len(extra_in_members),
        ],
    }
)

display(coverage)

if len(missing_in_members):
    display(pd.Series(missing_in_members, name="missing_clan_id").head(20))

if len(extra_in_members):
    display(pd.Series(extra_in_members, name="unused_member_clan_id").head(20))

,metric,count
0,unique clans in matches,48576
1,unique clans in members,48576
2,match clans missing in members,0
3,member clans not used in matches,0


In [16]:
roster_sizes = (
    members.groupby("clan_id")["user_id"]
    .nunique()
    .rename("n_members")
    .reset_index()
)

match_roster_sizes = pd.DataFrame({"clan_id": match_clans}).merge(
    roster_sizes,
    on="clan_id",
    how="left",
)

size_summary = (
    match_roster_sizes["n_members"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("n_members")
    .reset_index(name="n_clans")
)

display(size_summary)

display(match_roster_sizes.loc[match_roster_sizes["n_members"] != 6].head(20))

,n_members,n_clans
0,6,48576


,clan_id,n_members


## Repeated entities

If clans or users repeat, they could support history-based features. If they do not, the problem is mostly cross-sectional: each match has to be predicted from the current roster snapshot.


In [17]:
user_counts = (
    members["user_id"]
    .value_counts()
    .rename_axis("user_id")
    .reset_index(name="n_rows")
    .sort_values(["n_rows", "user_id"], ascending=[False, True])
    .reset_index(drop=True)
)

user_clan_counts = (
    members.groupby("user_id")["clan_id"]
    .nunique()
    .rename("n_unique_clans")
    .reset_index()
    .sort_values(["n_unique_clans", "user_id"], ascending=[False, True])
    .reset_index(drop=True)
)

clan_mentions = pd.concat(
    [
        matches["clan_1_id"].rename("clan_id"),
        matches["clan_2_id"].rename("clan_id"),
    ],
    ignore_index=True,
)

clan_match_counts = (
    clan_mentions
    .value_counts()
    .rename_axis("clan_id")
    .reset_index(name="n_match_mentions")
    .sort_values(["n_match_mentions", "clan_id"], ascending=[False, True])
    .reset_index(drop=True)
)

repeat_summary = pd.Series(
    {
        "member rows": len(members),
        "unique users": members["user_id"].nunique(),
        "users with multiple rows": int((user_counts["n_rows"] > 1).sum()),
        "users appearing in multiple clans": int((user_clan_counts["n_unique_clans"] > 1).sum()),
        "match rows": len(matches),
        "unique match clans": clan_match_counts["clan_id"].nunique(),
        "clans appearing in multiple matches": int((clan_match_counts["n_match_mentions"] > 1).sum()),
    },
    name="value",
)

display(repeat_summary.to_frame())

,value
member rows,291456
unique users,291456
users with multiple rows,0
users appearing in multiple clans,0
match rows,24288
unique match clans,48576
clans appearing in multiple matches,0


## Clan-side point table

The next checks use one row per clan appearance. This is more convenient than the match table for asking whether a clan's observed points are on the same scale as its current roster snapshot.


In [18]:
clan_features = (
    members.groupby("clan_id")
    .agg(
        n_members=("user_id", "nunique"),
        current_multiplier_sum=("clan_multiplier", "sum"),
        multiplier_mean=("clan_multiplier", "mean"),
        multiplier_max=("clan_multiplier", "max"),
    )
    .reset_index()
)

clan1_rows = pd.DataFrame(
    {
        "match_id": matches.index,
        "side": "clan_1",
        "clan_id": matches["clan_1_id"],
        "opponent_id": matches["clan_2_id"],
        "observed_points": matches["clan_1_points"],
        "opponent_points": matches["clan_2_points"],
        "won": matches["clan_winner"].eq(1),
    }
)

clan2_rows = pd.DataFrame(
    {
        "match_id": matches.index,
        "side": "clan_2",
        "clan_id": matches["clan_2_id"],
        "opponent_id": matches["clan_1_id"],
        "observed_points": matches["clan_2_points"],
        "opponent_points": matches["clan_1_points"],
        "won": matches["clan_winner"].eq(2),
    }
)

clan_side = pd.concat([clan1_rows, clan2_rows], ignore_index=True)
clan_side = clan_side.merge(clan_features, on="clan_id", how="left")

clan_side["point_margin"] = clan_side["observed_points"] - clan_side["opponent_points"]
clan_side["points_per_multiplier"] = (
    clan_side["observed_points"] / clan_side["current_multiplier_sum"].replace(0, np.nan)
)

print("clan-side rows:", clan_side.shape)
display(clan_side.sample(5, random_state=42))

clan-side rows: (48576, 13)


,match_id,side,clan_id,opponent_id,observed_points,opponent_points,won,n_members,current_multiplier_sum,multiplier_mean,multiplier_max,point_margin,points_per_multiplier
31664,7376,clan_2,clan_19360,clan_12871,47,37,True,6,16,2.666667,3,10,2.937500
41015,16727,clan_2,clan_44678,clan_1274,39,57,False,6,17,2.833333,3,-18,2.294118
8454,8454,clan_1,clan_27770,clan_39738,39,60,False,6,14,2.333333,3,-21,2.785714
16776,16776,clan_1,clan_37000,clan_28359,57,41,True,6,18,3.000000,3,16,3.166667
26960,2672,clan_2,clan_12725,clan_9716,75,24,True,6,18,3.000000,3,51,4.166667


## Multiplier scale check

This is not used as a rule for modeling. It is a sanity check on a tempting interpretation: if manager points were small fixed values multiplied by `clan_multiplier`, the current multiplier sum would roughly bound observed clan points. Large violations mean the multiplier column should be treated as a noisy roster feature, not as a complete accounting variable.


In [22]:
# two-leg multiplier check

clan_side["theoretical_max_two_legs"] = 6 * clan_side["current_multiplier_sum"]
clan_side["above_two_leg_max"] = clan_side["observed_points"] > clan_side["theoretical_max_two_legs"]

print("clan appearances:", len(clan_side))
print("above two-leg max:", clan_side["above_two_leg_max"].sum())


display(
    clan_side.loc[
        clan_side["above_two_leg_max"],
        [
            "match_id",
            "side",
            "clan_id",
            "observed_points",
            "current_multiplier_sum",
            "theoretical_max_two_legs",
        ],
    ].head(5)
)

clan appearances: 48576
above two-leg max: 427


,match_id,side,clan_id,observed_points,current_multiplier_sum,theoretical_max_two_legs
10903,10903,clan_1,clan_2341,45,7,42
13576,13576,clan_1,clan_6708,50,8,48
16410,16410,clan_1,clan_13499,57,9,54
16692,16692,clan_1,clan_9532,57,9,54
16760,16760,clan_1,clan_6001,57,9,54


## Multiplier consistency edge case

I initially flagged a small number of clan appearances where the observed points exceeded the theoretical two-leg maximum implied by the multiplier values in `member_stats_training.csv`. This looked like a possible data issue, so I contacted the Nordeus team for clarification.

They confirmed that this is an expected edge case from game mechanics not explicitly described in the challenge statement: `member_stats_training.csv` is a real pre-match snapshot, but some users can increase their Association multiplier after the snapshot is taken and before or between Association matches through an in-game event reward. Other values such as training bonuses or team quality can also change after the snapshot.